## About:

### This notebook contains data generation pipeline using huggingface models using AutoModelForCausalLM. The questions are picked from LSST forum page. Multiple models can be used just by adding its huggingface identifier in the list named model_list below.

In [28]:
# Install dependencies
!pip install transformers pandas torch -q
!pip install openpyxl


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [29]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import pandas as pd
import torch
from typing import List


# device = 0 if torch.cuda.is_available() else -1


In [30]:
def load_model_pipeline(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        trust_remote_code=True  # <-- This is essential
    )
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=device)
    return pipe


In [31]:
# Main function: Generate responses from multiple models
def generate_from_models(questions: List[str], model_names: List[str], max_new_tokens: int = 100):
    results = {'question': questions}
    for model_name in model_names:
        print(f"Loading model: {model_name}")
        pipe = load_model_pipeline(model_name)

        outputs = []
        for q in questions:
            response = pipe(q, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
            outputs.append(response[0]['generated_text'].strip())
        
        col_name = model_name.split('/')[-1]  # Use last part of model string as column name
        results[col_name] = outputs
    
    df = pd.DataFrame(results)
    return df


In [32]:
file_path = "data/lsst_forum_responses_5.xlsx"
df = pd.read_excel(file_path)

print(df.columns)

Index(['category_id', 'context', 'question', 'question_author',
       'question_date', 'answer', 'answer_author', 'answer_date',
       'primary_group_name', 'flair_name', 'flair_url', 'moderator', 'admin',
       'staff', 'is_accepted_answer'],
      dtype='object')


In [33]:
questions = df['question'].dropna().tolist()

model_list = [
    "allenai/OLMo-1B",
    "allenai/OLMo-1B-0724-hf",
    "allenai/OLMo-2-0425-1B-Instruct"
]

df_outputs = generate_from_models(questions, model_list, max_new_tokens=200)

df_outputs.head()


Loading model: allenai/OLMo-1B


Device set to use cpu


Loading model: allenai/OLMo-1B-0724-hf


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 61.04it/s]
Device set to use cpu


Loading model: allenai/OLMo-2-0425-1B-Instruct


Device set to use cpu


,question,OLMo-1B,OLMo-1B-0724-hf,OLMo-2-0425-1B-Instruct
0,"Hi, \nI’m following this tutorial: The LSST S...","Hi, \nI’m following this tutorial: The LSST S...","Hi, \nI’m following this tutorial: The LSST S...","Hi, \nI’m following this tutorial: The LSST S..."
1,I have the following C++ class : \n class CcdI...,I have the following C++ class : \n class CcdI...,I have the following C++ class : \n class CcdI...,I have the following C++ class : \n class CcdI...
2,Question on how forced photometry will be run ...,Question on how forced photometry will be run ...,Question on how forced photometry will be run ...,Question on how forced photometry will be run ...
3,"Hi there, \n Is there some way I find out what...","Hi there, \n Is there some way I find out what...","Hi there, \n Is there some way I find out what...","Hi there, \n Is there some way I find out what..."
4,I’m having trouble building FFTW with texinfo ...,I’m having trouble building FFTW with texinfo ...,I’m having trouble building FFTW with texinfo ...,I’m having trouble building FFTW with texinfo ...


In [35]:
# Save to CSV
df_outputs.to_csv("olmo_generation_output.csv", index=False)
